# Module SC9 - K-Fold Cross-Validation

Run on Kaggle with the supplied helper, image dataset, and SC1 configuration. Save a completed run's output for downstream analysis.

The source defaults to **three folds** and pools predictions from item-disjoint held-out test folds. The supplied pooled results cover the **historical 457-item cohort**, rather than the current 485-item dataset. The saved archive does not include enough fold metadata to verify the executed fold count. A rerun uses the item count in its attached manifests; record that dataset release separately.

**Backbone:** `convnext_small.fb_in22k_ft_in1k` by default. `MINORITY_BOOST` enables an optional rare-class-weighting recipe. Its availability does not establish that a separate comparison was completed.

Item identifiers define the folds. Augmented rows are included only in training folds, while validation and test use original image rows. Verify dataset identity and grouping when attaching new inputs.

Training runs the backbone once per fold. Runtime depends on the GPU, recipe, and attached dataset; use SC20 for the separate self-resuming five-fold workflow.

**Inputs:** `thesis-common-10c`, the image dataset, and SC1 `config.json`.
**Outputs:** `kfold_<tag>_metrics.json`, `kfold_<tag>_logits.npz`, `kfold_<tag>_ci.json`, and `kfold_<tag>_decision.json`.


In [ ]:

import sys, glob, os, json
import numpy as np
# Provenance bootstrap: explicit content identity; no timestamp/size selection.
import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  # T.find_one-only paths; generic cached artifact conflicts still require detaching.
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one
# Original local find()/find_all() consumers cannot obey T.find_one overrides.
# Reject conflicting standard cached artifacts up front instead of letting those
# consumers silently choose a same-name SC7/SC26 or stale output.
_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS), '| epochs:', T.EPOCHS)


In [ ]:

# ---- configuration ----
BACKBONE = "convnext_small.fb_in22k_ft_in1k"   # overall-best single backbone (material/threat winner).
                                               # For K=5 within the 12h cap, switch to "resnet50".
K        = 3              # 3 folds of convnext_small ~= 9-11h (fits Kaggle 12h cap). resnet50 supports K=5.
VAL_FRAC = 0.12           # slice of each fold's training items held out for early stopping
MINORITY_BOOST = False    # False = headline recipe (run FIRST: validates the reported model).
                          # True  = targeted rare-class recipe (run SECOND, compare hazardous_chemical/composite F1).

_SHORT = {"convnext_small.fb_in22k_ft_in1k": "convnext_small",
          "vit_small_patch14_dinov2.lvd142m": "dinov2"}
TAG = _SHORT.get(BACKBONE, BACKBONE) + ("_mb" if MINORITY_BOOST else "")
print(f"K-fold CV: backbone={BACKBONE}  K={K}  val_frac={VAL_FRAC}  minority_boost={MINORITY_BOOST}")
print(f"output tag = {TAG}  | epochs/fold={T.EPOCHS}  | approx {K} training runs of {BACKBONE}")


### (optional) targeted rare-class recipe — monkeypatch when `MINORITY_BOOST=True`

In [ ]:

# ---- (optional) targeted rare-class recipe — only active when MINORITY_BOOST=True ----
# The standard recipe ALREADY uses class-balanced focal + inverse-frequency oversampling, yet
# hazardous_chemical/composite (~21 items each, ~14 per training fold) stay hard. This run pushes
# harder so we can MEASURE — cross-validated over all 21 items — whether method or data is the limit.
# Honest expectation: small lift; the bottleneck is quantity. We monkeypatch (don't edit thesis_common).
if MINORITY_BOOST:
    import numpy as np, torch
    _orig_focal = T.focal_loss
    def _focal_g3(logits, target, gamma=3.0, alpha=None):      # harder focusing on hard/rare examples
        return _orig_focal(logits, target, gamma=gamma, alpha=alpha)
    T.focal_loss = _focal_g3
    T.LOSS_W = {**T.LOSS_W, "material": 2.0}                    # up-weight the material head (was 1.0)
    def _minority_sampler(df):                                 # push the smallest classes beyond plain 1/count
        counts = df["y_material"].value_counts().to_dict()
        rare = {m for m, c in counts.items() if c <= 30}        # e_waste / textile / composite / hazardous_chemical
        w = df["y_material"].map(lambda m: (2.0 if m in rare else 1.0) / counts[m]).to_numpy()
        return torch.utils.data.WeightedRandomSampler(
            torch.as_tensor(w, dtype=torch.double), num_samples=len(df), replacement=True)
    T.make_sampler = _minority_sampler
    print("MINORITY_BOOST ON: focal gamma=3.0, material loss weight x2, 2x oversampling for <=30-item classes.")
else:
    print("MINORITY_BOOST OFF: standard headline recipe (cross-validates the reported model).")


### K-fold training loop (trains the backbone K times, pools held-out logits)

In [ ]:

import torch, numpy as np, pandas as pd, json
from sklearn.model_selection import StratifiedKFold

ROOT = T.find_image_root()
df_tr, df_va, df_te = T.load_manifests(ROOT)
alldf = pd.concat([df_tr, df_va, df_te], ignore_index=True)
print("pooled rows:", len(alldf), "| items:", alldf.item_id.nunique(),
      "| sources:", alldf.source.value_counts().to_dict())

# one row per item (from ORIGINAL rows) to drive a stratified split by material
orig = alldf[alldf.source == "original"].drop_duplicates("item_id")
items = orig.item_id.to_numpy()
strat = orig.y_material.to_numpy()
print("items entering CV:", len(items))

skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=T.SEED)
HEADS = ["material","condition","route","threat"]
P  = {k: [] for k in HEADS}; PY = {k: [] for k in HEADS}; PIT = []

for fold,(tr_idx, te_idx) in enumerate(skf.split(items, strat)):
    te_items = set(items[te_idx])
    rng = np.random.default_rng(T.SEED + fold)
    rem = rng.permutation(items[tr_idx])
    n_val = max(int(len(rem)*VAL_FRAC), K)
    va_items, trn_items = set(rem[:n_val]), set(rem[n_val:])
    tr_f = alldf[alldf.item_id.isin(trn_items)]                                    # aug + original
    va_f = alldf[(alldf.item_id.isin(va_items)) & (alldf.source=="original")]      # original only
    te_f = alldf[(alldf.item_id.isin(te_items)) & (alldf.source=="original")]      # original only
    T.check_leakage(tr_f, va_f, te_f)
    print(f"\n=== FOLD {fold+1}/{K} | train items {len(trn_items)} (rows {len(tr_f)}) "
          f"| val {len(va_items)} | test {len(te_items)} (rows {len(te_f)})")
    model, hist = T.train_backbone(BACKBONE, tr_f, va_f, te_f)
    _, _, te_loader = T.make_loaders(tr_f, va_f, te_f)
    o, y, it = T.evaluate_logits(model, te_loader)
    for k in HEADS: P[k].append(o[k]); PY[k].append(y[k])
    PIT += list(it)
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

P  = {k: np.concatenate(v) for k,v in P.items()}
PY = {k: np.concatenate(v) for k,v in PY.items()}
PIT = np.array(PIT)
print("\npooled held-out image rows:", len(PIT), "| unique items:", len(set(PIT.tolist())))


### Pooled per-head + per-class evaluation over all items

In [ ]:

# ---- pooled report over ALL items (each tested once) ----
rep = T.full_report(P, PY, PIT)
json.dump(rep, open(f"kfold_{TAG}_metrics.json","w"), indent=2)
np.savez_compressed(f"kfold_{TAG}_logits.npz",
    **{f"test_log_{k}": P[k]  for k in P},
    **{f"test_y_{k}":   PY[k] for k in PY}, test_items=PIT.astype(str))
s = rep["scalars"]
print("pooled items:", rep["n_items"], "(target 457)")
print("MATERIAL  acc=%.3f  macroF1=%.3f" % (s["material"]["acc"], s["material"]["macro_f1"]))
print("ROUTE     acc=%.3f  macroF1=%.3f" % (s["route"]["acc"], s["route"]["macro_f1"]))
print("THREAT    QWK=%.3f  MAE=%.3f  acc=%.3f" % (s["threat"]["qwk"], s["threat"]["mae"], s["threat"]["acc"]))
print("CONDITION AUROC=%.3f" % s["condition"]["auroc"])
print("\nMATERIAL per-class F1 (worst first) — rare classes now have ~5x support vs the 46-item split:")
for k,v in sorted(rep["per_class"]["material"].items(), key=lambda kv: kv[1]["f1"]):
    print(f"   {k:20s} f1={v['f1']:.2f}  P={v['precision']:.2f} R={v['recall']:.2f}  n={v['support']}")
print("\nTHREAT per level:", rep["threat_per_level"])


### Pooled routing decision + hazard recall over all items (safety claim, with CI)

In [ ]:

# ---- pooled ROUTING over ALL items: policy + threat backstop -> cost / coverage / hazard recall (with CI) ----
# This evaluates the SAFETY claim over every Special-route item in the dataset, not just the 15 in the 46-item split.
import json, torch, numpy as np
agg, uniq, idx = T.aggregate_by_item(P, PIT)
lab   = T.item_labels(PY, PIT, idx)
mprob = T.softmax_np(agg["material"]); cprob = T.sigmoid_np(agg["condition"])
tprob = T.corn_class_probs(torch.tensor(agg["threat"])).numpy()
routes, _  = T.apply_policy(mprob, cprob, tprob)
routes     = np.asarray(routes, dtype=object)
true_route = lab["route"].astype(int)
base = T.decision_report(routes, true_route)
print("pooled routing over", base["n"], "items:",
      {k: (round(v, 4) if isinstance(v, float) else v) for k, v in base.items()})

rng = np.random.default_rng(T.SEED); n = len(uniq); ii = np.arange(n); hz, cst = [], []
for _ in range(5000):
    sel = rng.choice(ii, size=n, replace=True)
    d = T.decision_report(routes[sel], true_route[sel])
    hz.append(d["hazard_recall"]); cst.append(d["total_cost"]/d["n"])
def _ci(a): a = np.array(a, float); return [round(float(np.percentile(a, 2.5)), 4), round(float(np.percentile(a, 97.5)), 4)]
print(f"hazard_recall {base['hazard_recall']:.3f}  95% CI {_ci(hz)}")
print(f"mean cost/item {base['total_cost']/base['n']:.3f}  95% CI {_ci(cst)}")
json.dump({"pooled": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in base.items()},
           "hazard_recall_ci": _ci(hz), "cost_per_item_ci": _ci(cst)},
          open(f"kfold_{TAG}_decision.json", "w"), indent=2)
print("\nHazard recall here is over EVERY Special-route item in the dataset (the strongest form of the safety claim).")


### Bootstrap 95% confidence intervals on the pooled evaluation

In [ ]:

# ---- item-level bootstrap 95% CI on the pooled (~457-item) evaluation ----
import torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, cohen_kappa_score
agg, uniq, idx = T.aggregate_by_item(P, PIT)
lab = T.item_labels(PY, PIT, idx)
def _metrics(sel):
    mp = T.softmax_np(agg["material"][sel]).argmax(1)
    rp = T.softmax_np(agg["route"][sel]).argmax(1)
    cp = T.sigmoid_np(agg["condition"][sel])
    tp = T.corn_levels(torch.tensor(agg["threat"][sel])).numpy()
    ym,yr,yc,yt = (lab["material"][sel], lab["route"][sel],
                   lab["condition"][sel].astype(int), lab["threat"][sel])
    qwk = cohen_kappa_score(yt, tp, weights="quadratic", labels=list(range(T.NUM_THREAT))) if len(np.unique(yt))>1 else np.nan
    au  = roc_auc_score(yc, cp) if len(np.unique(yc))>1 else np.nan
    return {"material_acc":accuracy_score(ym,mp),
            "material_f1":f1_score(ym,mp,average="macro",labels=range(len(T.MATERIALS)),zero_division=0),
            "route_f1":f1_score(yr,rp,average="macro",labels=range(len(T.ROUTES)),zero_division=0),
            "threat_qwk":qwk, "condition_auroc":au}
n = len(uniq); ii = np.arange(n); rng = np.random.default_rng(T.SEED)
point = _metrics(ii); B = 5000; boot = {k:[] for k in point}
for _ in range(B):
    sel = rng.choice(ii, size=n, replace=True)
    for k,v in _metrics(sel).items(): boot[k].append(v)
ci = {}
for k in point:
    a = np.array(boot[k], float); a = a[~np.isnan(a)]
    lo,hi = np.percentile(a,[2.5,97.5])
    ci[k] = {"point":round(float(point[k]),4),"lo":round(float(lo),4),"hi":round(float(hi),4)}
    print(f"{k:16s} {point[k]:.3f}  95% CI [{lo:.3f}, {hi:.3f}]")
json.dump({"n_items":n,"n_bootstrap":B,"ci":ci}, open(f"kfold_{TAG}_ci.json","w"), indent=2)
print("\nCompare these CIs to BOOTSTRAP_CI.md (single 46-item split) — they should be noticeably tighter.")
